# EDA: NFL Big Data Bowl 2027 (Vojtech Naar)

A first look at every table in `data/bdb2027.sqlite`.

**Before running:** build the database from the repo root with `python src/load_data.py` (see the README).

Sections: setup, table overview, loading, missing values, plots (players, games, combine, tracking), notes.

## 1. Setup

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)

# Same look for every chart: one color, light grid, no top/right borders
COLOR = "#2a78d6"
plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
# Find data/bdb2027.sqlite whether the notebook runs from src/eda_vojtech_naar/ or the repo root
DB_PATH = next(
    (p / "data" / "bdb2027.sqlite" for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "bdb2027.sqlite").exists()),
    None,
)
assert DB_PATH is not None, "Database not found. Run `python src/load_data.py` from the repo root first."

con = sqlite3.connect(DB_PATH)
print("Connected to", DB_PATH)

## 2. Overview of all tables

In [ ]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con
)["name"].tolist()

overview = pd.DataFrame({
    "table": tables,
    "rows": [pd.read_sql(f'SELECT COUNT(*) AS n FROM "{t}"', con)["n"][0] for t in tables],
    "columns": [len(pd.read_sql(f'PRAGMA table_info("{t}")', con)) for t in tables],
}).sort_values("rows", ascending=False).reset_index(drop=True)
overview

In [ ]:
ax = overview.sort_values("rows").plot.barh(x="table", y="rows", color=COLOR, legend=False)
ax.set_xscale("log")
ax.set_xlabel("Rows (log scale)")
ax.set_ylabel("")
ax.set_title("Rows per table")
plt.tight_layout()
plt.show()

## 3. Load the data

Tables under `MAX_ROWS` are loaded in full. Bigger ones (the tracking tables) are loaded as a
random sample so the notebook stays fast. Raise `MAX_ROWS` if your machine can handle it.

In [ ]:
MAX_ROWS = 500_000
SAMPLE_ROWS = 200_000

dfs = {}
for t, n in zip(overview["table"], overview["rows"]):
    if n <= MAX_ROWS:
        dfs[t] = pd.read_sql(f'SELECT * FROM "{t}"', con)
    else:
        dfs[t] = pd.read_sql(
            f'SELECT * FROM "{t}" WHERE rowid IN '
            f'(SELECT rowid FROM "{t}" ORDER BY RANDOM() LIMIT {SAMPLE_ROWS})', con
        )
    print(f"{t:<28} loaded {len(dfs[t]):>9,} of {n:>11,} rows")

In [ ]:
# Peek at the first rows of every table
for t, df in dfs.items():
    print(f"=== {t} ===")
    display(df.head(3))

## 4. Missing values

In [ ]:
missing = pd.DataFrame([
    {"table": t, "column": c, "missing_pct": round(100 * df[c].isna().mean(), 2)}
    for t, df in dfs.items() for c in df.columns
])
missing = missing[missing["missing_pct"] > 0].sort_values("missing_pct", ascending=False)
print(f"{len(missing)} columns have missing values")
missing.head(30)

## 5. Table-by-table plots

Each cell checks that the columns it needs exist, so it skips cleanly if a name is different
in your copy of the data. Check the `head()` output in section 3 if something is skipped.

In [ ]:
def hist(df, col, title, bins=40):
    """Histogram of one numeric column."""
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    fig, ax = plt.subplots()
    ax.hist(s, bins=bins, color=COLOR, edgecolor="white", linewidth=0.5)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()


def top_counts(df, col, title, n=15):
    """Horizontal bar chart of the n most common values in a column."""
    counts = df[col].value_counts().head(n).sort_values()
    fig, ax = plt.subplots(figsize=(9, max(3, 0.3 * len(counts))))
    ax.barh(counts.index.astype(str), counts.values, color=COLOR)
    ax.set_title(title)
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.show()


def first_col(df, *candidates):
    """Return the first candidate column name that exists in df, else None."""
    return next((c for c in candidates if c in df.columns), None)

### 5a. Players

In [ ]:
players = dfs.get("players", pd.DataFrame())
print(players.shape)

pos = first_col(players, "position", "pos", "officialPosition")
if pos:
    top_counts(players, pos, "Players by position")

for col in ["height", "weight", "age"]:
    c = first_col(players, col, col.capitalize())
    if c and pd.api.types.is_numeric_dtype(players[c]):
        hist(players, c, f"Player {col}")

### 5b. Games

In [ ]:
games = dfs.get("games", pd.DataFrame())
print(games.shape)

season = first_col(games, "season", "year")
if season:
    top_counts(games, season, "Games per season")

week = first_col(games, "week")
if week:
    per_week = games[week].value_counts().sort_index()
    fig, ax = plt.subplots()
    ax.bar(per_week.index.astype(str), per_week.values, color=COLOR)
    ax.set_title("Games per week (all seasons)")
    ax.set_xlabel("Week")
    ax.set_ylabel("Games")
    plt.tight_layout()
    plt.show()

### 5c. Combine results

In [ ]:
combine = dfs.get("combine_results", pd.DataFrame())
print(combine.shape)

# Histogram for each numeric drill result (skip ID / year columns)
skip = ("id", "year", "season")
num_cols = [c for c in combine.select_dtypes("number").columns
            if not any(k in c.lower() for k in skip)]
for c in num_cols[:4]:
    hist(combine, c, f"Combine: {c}")

### 5d. Tracking data (sampled)

Columns follow the usual Big Data Bowl format: `x`, `y` (field position in yards),
`s` (speed, yd/s), `a` (acceleration, yd/s²), `dis` (distance since the last frame), `dir` (direction).

In [ ]:
tracking_tables = [t for t in dfs if "tracking" in t]
for t in tracking_tables:
    df = dfs[t]
    print(f"=== {t}: {df.shape} ===")
    for c, label in [("s", "speed (yd/s)"), ("a", "acceleration (yd/s²)")]:
        if c in df.columns:
            hist(df, c, f"{t}: {label}", bins=60)

In [ ]:
# Where on the field are players? (one game-tracking table, sampled)
game_tables = [t for t in tracking_tables if t.startswith("game_tracking")]
if game_tables and {"x", "y"} <= set(dfs[game_tables[-1]].columns):
    t = game_tables[-1]
    df = dfs[t]
    fig, ax = plt.subplots(figsize=(12, 5.5))
    hb = ax.hexbin(df["x"], df["y"], gridsize=60, cmap="Blues", mincnt=1)
    fig.colorbar(hb, ax=ax, label="Tracking points")
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 53.3)
    ax.set_aspect("equal")
    ax.set_xlabel("x (yards, end zones at 0-10 and 110-120)")
    ax.set_ylabel("y (yards)")
    ax.set_title(f"{t}: player positions on the field")
    ax.grid(False)
    plt.tight_layout()
    plt.show()

## 6. Notes and next steps

Write down what stood out while running this:

- Which tables join together, and on which keys (for example `nfl_id`, `game_id`, `play_id`)?
- Columns with a lot of missing data, and whether to drop or fill them
- Outliers in speed, acceleration, or combine results that look like data errors
- Ideas for the project question

In [ ]:
con.close()